In [1]:
import torch

print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))

GPU available: True
Device: Tesla T4


In [2]:
import os, zipfile, urllib.request

DATA_DIR = "/kaggle/working/tiny-imagenet-200"

if not os.path.exists(DATA_DIR):
    url = "http://cs231n.stanford.edu/tiny-imagenet-200.zip"
    urllib.request.urlretrieve(url, "/kaggle/working/tiny-imagenet-200.zip")

    with zipfile.ZipFile("/kaggle/working/tiny-imagenet-200.zip", 'r') as zip_ref:
        zip_ref.extractall("/kaggle/working/")

In [3]:
import os, shutil

val_dir = os.path.join(DATA_DIR, "val")
images_dir = os.path.join(val_dir, "images")
annotations = os.path.join(val_dir, "val_annotations.txt")

with open(annotations) as f:
    for line in f:
        img, cls = line.split()[:2]
        os.makedirs(os.path.join(val_dir, cls), exist_ok=True)
        shutil.move(
            os.path.join(images_dir, img),
            os.path.join(val_dir, cls, img)
        )

shutil.rmtree(images_dir)

In [ ]:
# importing all necessary packages for building model and performance metrics
import time
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms

from torch.utils.data import DataLoader
from torchvision.models import resnet50
from tqdm import tqdm
from torch.cuda.amp import autocast, GradScaler

In [ ]:
# define data transformations on training and validation set
def get_loaders(data_dir=DATA_DIR):
    train_tf = transforms.Compose([
        transforms.RandomResizedCrop(64, scale=(0.6, 1.0)),
        transforms.RandomHorizontalFlip(),
        transforms.ColorJitter(0.4, 0.4, 0.4, 0.1),
        transforms.ToTensor(),
        transforms.Normalize(
            (0.4802, 0.4481, 0.3975),
            (0.2302, 0.2265, 0.2262)
        ),
        transforms.RandomErasing(p=0.25)
    ])

    val_tf = transforms.Compose([
        transforms.Resize(64),
        transforms.CenterCrop(64),
        transforms.ToTensor(),
        transforms.Normalize(
            (0.4802, 0.4481, 0.3975),
            (0.2302, 0.2265, 0.2262)
        ),
    ])

    train_ds = torchvision.datasets.ImageFolder(
        os.path.join(data_dir, "train"),
        transform=train_tf
    )
    val_ds = torchvision.datasets.ImageFolder(
        os.path.join(data_dir, "val"),
        transform=val_tf
    )

    train_loader = DataLoader(
        train_ds, batch_size=128, shuffle=True,
        num_workers=0, pin_memory=True
    )
    val_loader = DataLoader(
        val_ds, batch_size=256, shuffle=False,
        num_workers=0
    )

    return train_loader, val_loader

In [ ]:
# perform data augmentation
def rand_bbox(size, lam):
    W, H = size[2], size[3]
    cut_rat = np.sqrt(1. - lam)
    cut_w, cut_h = int(W * cut_rat), int(H * cut_rat)

    cx, cy = np.random.randint(W), np.random.randint(H)

    bbx1 = np.clip(cx - cut_w // 2, 0, W)
    bby1 = np.clip(cy - cut_h // 2, 0, H)
    bbx2 = np.clip(cx + cut_w // 2, 0, W)
    bby2 = np.clip(cy + cut_h // 2, 0, H)

    return bbx1, bby1, bbx2, bby2


def cutmix(x, y, alpha=1.0):
    indices = torch.randperm(x.size(0)).to(x.device)
    lam = np.random.beta(alpha, alpha)

    bbx1, bby1, bbx2, bby2 = rand_bbox(x.size(), lam)
    x[:, :, bbx1:bbx2, bby1:bby2] = x[indices, :, bbx1:bbx2, bby1:bby2]

    lam = 1 - ((bbx2 - bbx1) * (bby2 - bby1) / (x.size(-1) * x.size(-2)))

    return x, y, y[indices], lam

In [ ]:
# create evaluation fcn
def evaluate(model, loader, device):
    model.eval()
    correct = total = 0

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            pred = out.argmax(1)
            correct += (pred == y).sum().item()
            total += y.size(0)

    return 100 * correct / total

In [ ]:
import psutil

# functions for tracking model size, GPU/CPU usage
def get_model_size(model):
    param_size = sum(p.nelement() * p.element_size() for p in model.parameters())
    buffer_size = sum(b.nelement() * b.element_size() for b in model.buffers())
    return (param_size + buffer_size) / 1024**2  # MB

def get_cpu_usage():
    return psutil.cpu_percent(interval=None)

def get_gpu_usage():
    if torch.cuda.is_available():
        return torch.cuda.memory_allocated() / 1024**2  # MB
    return 0

In [ ]:
def train():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    train_loader, val_loader = get_loaders()

    # using pre-trained weights from ImageNet
    model = resnet50(weights="IMAGENET1K_V1")
    model.fc = nn.Linear(model.fc.in_features, 200)
    model.to(device)

    # define optimizer
    optimizer = optim.SGD(
        model.parameters(), lr=0.1,
        momentum=0.9, weight_decay=1e-4
    )

    # define optimizer
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    scaler = GradScaler()

    best_acc = 0
    patience = 20
    counter = 0

    total_start_time = time.time()

    print("\n===== TRAINING START =====")

    for epoch in range(50):
        # tracking time
        epoch_start = time.time()

        model.train()
        total_loss = 0

        cpu_usage_epoch = []
        gpu_usage_epoch = []

        for x, y in tqdm(train_loader):
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()

            use_cutmix = np.random.rand() < 0.3
            if use_cutmix:
                x, y_a, y_b, lam = cutmix(x, y)

            with autocast():
                out = model(x)
                if use_cutmix:
                    loss = lam * criterion(out, y_a) + (1 - lam) * criterion(out, y_b)
                else:
                    loss = criterion(out, y)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            total_loss += loss.item()

            cpu_usage_epoch.append(get_cpu_usage())
            if torch.cuda.is_available():
                gpu_usage_epoch.append(get_gpu_usage())

        scheduler.step()

        epoch_time = time.time() - epoch_start
        val_acc = evaluate(model, val_loader, device)

        # basic logging
        print(f"\nEpoch {epoch+1:02d}")
        print(f"Loss: {total_loss:.2f}")
        print(f"Val Acc: {val_acc:.2f}%")
        print(f"Epoch Time: {epoch_time:.2f}s")

        # print avg cpu/gpu usage every 10 epochs
        if (epoch + 1) % 10 == 0:
            print(f"Avg CPU Usage: {sum(cpu_usage_epoch)/len(cpu_usage_epoch):.2f}%")
            if gpu_usage_epoch:
                print(f"Avg GPU Memory: {sum(gpu_usage_epoch)/len(gpu_usage_epoch):.2f} MB")

        # early stopping logic
        if val_acc > best_acc:
            best_acc = val_acc
            torch.save(model.state_dict(), "/kaggle/working/best_model.pth")
            counter = 0
        else:
            counter += 1

        if counter >= patience:
            print("Early stopping")
            break

    total_time = time.time() - total_start_time

    print("\n===== TRAINING COMPLETE =====")
    print(f"Total Training Time: {total_time/60:.2f} minutes")
    print(f"Best Validation Accuracy: {best_acc:.2f}%")
    # print model size after training completes
    print(f"Model Size: {get_model_size(model):.2f} MB")

    return model, val_loader

In [ ]:
# train model
model, val_loader = train()

Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 151MB/s] 
/tmp/ipykernel_55/1697043434.py:17: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()



===== TRAINING START =====


  0%|          | 0/782 [00:00<?, ?it/s]/tmp/ipykernel_55/1697043434.py:44: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
100%|██████████| 782/782 [04:36<00:00,  2.83it/s]



Epoch 01
Loss: 3625.11
Val Acc: 15.49%
Epoch Time: 276.53s


...


Epoch 48
Loss: 1377.99
Val Acc: 58.77%
Epoch Time: 240.46s


100%|██████████| 782/782 [03:59<00:00,  3.26it/s]



Epoch 49
Loss: 1376.42
Val Acc: 59.24%
Epoch Time: 239.92s


100%|██████████| 782/782 [03:58<00:00,  3.27it/s]



Epoch 50
Loss: 1427.44
Val Acc: 59.22%
Epoch Time: 238.98s
Avg CPU Usage: 27.34%
Avg GPU Memory: 302.60 MB

===== TRAINING COMPLETE =====
Total Training Time: 222.50 minutes
Best Validation Accuracy: 59.24%
Model Size: 91.44 MB


In [ ]:
print("\n===== INT8 QUANTIZATION =====")
model_cpu = model.to("cpu")
model_cpu.eval()

start_time = time.time()

quantized_model = torch.quantization.quantize_dynamic(
    model_cpu,
    {nn.Linear},
    dtype=torch.qint8
)

acc = evaluate(quantized_model, val_loader, "cpu")

print(f"INT8 Accuracy: {acc:.2f}%")
print(f"INT8 Model Size: {get_model_size(quantized_model):.2f} MB")
print(f"INT8 CPU Usage: {get_cpu_usage():.2f}%")
print(f"INT8 GPU Usage: {get_gpu_usage():.2f} MB")
print(f"Quantization Time: {time.time() - start_time:.2f}s")


===== INT8 QUANTIZATION =====


/tmp/ipykernel_55/3556492916.py:8: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  quantized_model = torch.quantization.quantize_dynamic(


INT8 Accuracy: 59.23%
INT8 Model Size: 89.88 MB
INT8 CPU Usage: 38.80%
INT8 GPU Usage: 17.25 MB
Quantization Time: 101.13s
